# [Chapter 6 — SIR_I Analysis, §6.4-6.5] The Invasion–Burden Partition (Theorem 4.X)

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 6 — SIR_I Analysis
**Considerations developed:** 6 (R_0), 7 (equilibria + stability), 10 (sensitivity)
**Estimated runtime:** ≤ 30 seconds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Demonstrates the book's invasion–burden partition: $R_0$ governs whether the epidemic can invade (the threshold $R_0 > 1$), while a different parameter combination governs the endemic burden $I^*$. These are dual but distinct phenomena; the notebook visualizes both.

## What you should already know
Chapter 6 R_0 notebooks.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS
from shared.parameters import baseline_chapter_05
from shared.seeds import set_seed_chapter_06

set_seed_chapter_06()
book_style()


## The invasion–burden distinction

The book's Theorem `thm:invasion-burden` formalizes a property that is widely overlooked:

- **Invasion** is governed by $R_0$. The epidemic can establish itself if and only if $R_0 > 1$.
- **Burden** at endemic equilibrium is governed by a *different* combination of parameters. For SIR with vital dynamics, the endemic infected fraction $I^* / N$ depends on $R_0$, the demographic turnover $\mu$, and other parameters — not just $R_0$ alone.

The chapter's epigraph captures this: *"$R_0 > 1$ tells you the epidemic can begin. It says nothing about how it ends."*

In [ ]:
# For SIR with vital dynamics, the endemic equilibrium I*/N depends on parameters differently
# Equations from book §4.5 (with mu = demographic turnover):
#   S* = N / R_0
#   I* = (mu N / (mu + 1/tau_R)) * (1 - 1/R_0)
#   R* = N - S* - I*

# Sweep R_0 by varying c_I; track invasion (R_0) and burden (I*/N)
c_I_range = np.linspace(0.5, 5, 50) * 8  # span around baseline c_I = 8

beta = 0.025
tau_R = 10.0
mu = 1.0 / 365.0  # 1/year demographic turnover
N = 1.0

R0_array = c_I_range * beta * tau_R
S_star = N / R0_array
I_star = (mu * N / (mu + 1/tau_R)) * (1 - 1/R0_array)
I_star = np.where(R0_array > 1, I_star, 0)  # I* = 0 when R_0 < 1

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(c_I_range, R0_array, color=BOOK_COLORS['primary'], lw=1.5)
ax.axhline(1.0, color=BOOK_COLORS['negative'], ls='--', lw=1, alpha=0.7, label='Invasion threshold $R_0 = 1$')
ax.fill_between(c_I_range, 0, R0_array, where=(R0_array<1), alpha=0.1, color=BOOK_COLORS['negative'])
ax.fill_between(c_I_range, 0, R0_array, where=(R0_array>=1), alpha=0.1, color=BOOK_COLORS['positive'])
ax.set_xlabel('$c_I$ (contact rate)')
ax.set_ylabel('$R_0$')
ax.set_title('Invasion: $R_0 = c_I \\beta \\tau_R$')
ax.legend()

ax = axes[1]
ax.plot(R0_array, I_star, color=BOOK_COLORS['infectious'], lw=1.5)
ax.axvline(1.0, color=BOOK_COLORS['negative'], ls='--', lw=1, alpha=0.7, label='Invasion threshold')
ax.set_xlabel('$R_0$')
ax.set_ylabel('$I^* / N$')
ax.set_title('Burden: $I^*/N$ depends on $\\mu, \\tau_R$ — not just $R_0$')
ax.legend()

plt.tight_layout()
plt.show()


## Why this distinction matters

A common misconception: "$R_0 = 1.5$ means a moderate outbreak, $R_0 = 5$ means a severe one." This is partially true (larger $R_0$ means harder to control, larger initial growth rate) but misses a crucial point:

The **endemic burden** $I^*$ depends on parameters that don't appear in $R_0$:
- Demographic turnover $\mu$
- Recovery time $\tau_R$
- Birth-mortality balance

Two diseases with the same $R_0$ can have endemic burdens differing by orders of magnitude (e.g., measles vs influenza). Policy decisions targeting "lower $R_0$" are addressing invasion control; policies targeting "lower endemic prevalence" need to address the burden equation.

This is **Consideration 6** elaborated. Chapter 10 develops the sensitivity-analysis tools (Chapter 10 in the printed book) to formalize this further: the parameter rankings for invasion and burden are *different*, and pretending they're the same misleads policy.